# Notebook 11: Reinsurance sensitivity and capital

This notebook applies identical reinsurance programs to the three
validated Notebook 10 dependence cases:

- **I0_PHASE1_INDEPENDENT**: exact Phase 1 control;
- **C1_ALDEA22_SUBDUCTION**: primary spatial-correlation case;
- **C2_GODA_ATKINSON09**: correlation-model sensitivity.

The frozen benchmark is the Phase 1 occurrence excess-of-loss layer:
**$18.811 million attachment, $61.838 million limit, and 100 percent
participation**. The same terms are applied to every case.

The notebook also evaluates common occurrence and annual aggregate
design grids, a standalone annual aggregate stop-loss, and a stacked
occurrence-plus-aggregate program. It reports retained and ceded AAL,
AEP, OEP, PML, VaR, TVaR, capital efficiency, required limits,
diversification, paired Monte Carlo uncertainty, break-even premium,
and RAROC assumption grids.

No observed insurance price, expense ratio, reinsurance rate, or
regulatory-capital assumption is introduced. RAROC is reported only
over an explicit scenario grid.


In [ ]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import math
import os
from pathlib import Path
import sys
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display


PIPELINE_VERSION = "notebook11_reinsurance_capital_v1"
SCHEMA_VERSION = "notebook11_reinsurance_capital_handoff_v1"
EXPECTED_CATALOG_YEARS = 2_000_000
EXPECTED_OCCURRENCES = 10_630
EXPECTED_SITES = 470
EXPECTED_OCCUPIED_YEARS = 10_593
EXPECTED_ZERO_EVENT_YEARS = 1_989_407
ROW_TOLERANCE_USD = 2.0e-6
AGGREGATE_TOLERANCE_USD = 0.01
LIMIT_SEARCH_TOLERANCE_USD = 1_000.0
BOOTSTRAP_REPLICATES = int(
    os.environ.get("NOTEBOOK11_BOOTSTRAP_REPLICATES", "300")
)
BOOTSTRAP_SEED = 112_026
VERIFY_INPUT_HASHES = (
    os.environ.get("NOTEBOOK11_VERIFY_INPUT_HASHES", "1") == "1"
)
if BOOTSTRAP_REPLICATES < 2:
    raise ValueError("NOTEBOOK11_BOOTSTRAP_REPLICATES must be at least 2.")

FROZEN_LAYER = {
    "layer_scenario_id": "baseline_occurrence_xol_500yr_to_2500yr_oep_v1",
    "attachment_2022_usd": 18_811_083.54014544,
    "limit_2022_usd": 61_837_983.314918146,
    "exhaustion_2022_usd": 80_649_066.85506359,
    "participation": 1.0,
    "attachment_return_period_years": 500,
    "exhaustion_return_period_years": 2_500,
}
EXPECTED_PHASE1_REINSURANCE = {
    "triggering_occurrences": 4_005,
    "exhausting_occurrences": 800,
    "ceded_loss_total_2022_usd": 127_353_194_654.53214,
    "retained_loss_total_2022_usd": 118_605_925_280.12482,
    "ceded_aal_2022_usd": 63_676.59732726607,
    "retained_aal_2022_usd": 59_302.96264006241,
    "maximum_ceded_occurrence_loss_2022_usd": 61_837_983.314918146,
    "maximum_retained_occurrence_loss_2022_usd": 182_487_824.01776767,
    "maximum_ceded_aep_2022_usd": 90_413_665.12193382,
}
ATTACHMENT_RETURN_PERIODS = [100, 250, 500, 1_000]
EXHAUSTION_RETURN_PERIODS = [1_000, 2_500, 5_000, 10_000]
HEADLINE_RETURN_PERIODS = [100, 250, 500, 1_000, 2_500, 5_000, 10_000]
PREMIUM_MULTIPLES_OF_GROSS_AAL = [1.25, 1.5, 2.0, 2.5, 3.0]
EXPENSE_RATIOS = [0.20, 0.30, 0.40]
CEDED_PRICE_MULTIPLIERS = [1.0, 1.25, 1.5, 2.0]
EXPECTED_HASHES = {
    "phase1_reinsurance_formula": (
        "f347b96e186abd3a098e8597fbd5cb0ad318a7a58f56e567e3cbc28b17d7aea8"
    ),
    "notebook10_handoff": (
        "f9bb75c8e596ebdd21b2e4d5ca39c472074d4e79e189c5959c836f3c386f1fc4"
    ),
    "notebook10_validation": (
        "2a2ad99a4d2eb8dfa4de639ac730ab5041683ab5698032ccf958292fe13158de"
    ),
}


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "11_reinsurance_sensitivity_and_capital.ipynb").exists():
            return candidate
        if (
            (candidate / "tools" / "reinsurance_capital.py").exists()
            and (candidate / "10_correlated_damage_and_loss.ipynb").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not identify the seismic-correlation-insurance-loss repository root."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tools.reinsurance_capital import (
    CASE_PREFIXES,
    CONFIDENCE_LEVELS,
    RETURN_PERIODS,
    annual_program_metrics,
    annualize_occurrence_program,
    apply_aggregate_stop_loss,
    apply_occurrence_xol,
    diversification_from_sparse_units,
    empirical_pml,
    empirical_var_tvar,
    minimum_occurrence_limit_for_target,
    paired_bootstrap_differences,
    reconcile_annual_waterfall,
)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def sha256_lf_normalized_text(path: Path) -> str:
    # Hash text after canonical CRLF/CR to LF normalization.
    data = path.read_bytes().replace(b"\r\n", b"\n").replace(b"\r", b"\n")
    return hashlib.sha256(data).hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    data = (
        json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n"
    ).encode("utf-8")
    temporary.write_bytes(data)
    temporary.replace(path)


def write_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    data = frame.to_csv(index=False, lineterminator="\n").encode("utf-8")
    temporary.write_bytes(data)
    temporary.replace(path)


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as raw:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw,
            compresslevel=1,
            mtime=0,
        ) as compressed:
            with io.TextIOWrapper(
                compressed, encoding="utf-8", newline=""
            ) as text:
                frame.to_csv(
                    text,
                    index=False,
                    float_format="%.17g",
                    lineterminator="\n",
                )
    temporary.replace(path)


def project_relative_path(path: Path) -> str:
    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except ValueError as exc:
        raise ValueError(f"Path is outside the project root: {path}") from exc


def resolve_repository_path(value: object) -> Path:
    text = str(value).strip().replace("\\", "/")
    marker = "/data/"
    if marker in text.lower():
        start = text.lower().index(marker) + 1
        return PROJECT_ROOT.joinpath(*text[start:].split("/"))
    path = Path(text).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path.resolve()


def normalize_boolean_series(values: pd.Series) -> pd.Series:
    return (
        values.astype("string")
        .fillna("")
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


def add_check(
    rows: list[dict[str, object]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


NOTEBOOK10_METADATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "phase_2"
    / "notebook_10_correlated_damage_loss"
)
NOTEBOOK11_METADATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "phase_2"
    / "notebook_11_reinsurance_capital"
)
OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "phase_2"
    / "notebook_11_reinsurance_capital"
)
NOTEBOOK10_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "phase_2"
    / "notebook_10_correlated_damage_loss"
)
NOTEBOOK11_METADATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK10_HANDOFF_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_final_handoff.json"
NOTEBOOK10_VALIDATION_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_final_validation.csv"
NOTEBOOK10_PML_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_pml_table.csv"
BUILDING_LOSS_PATH = NOTEBOOK10_OUTPUT_DIR / "paired_damage_and_policy_loss.csv.gz"
EVENT_LOSS_PATH = NOTEBOOK10_OUTPUT_DIR / "paired_event_loss_summary.csv.gz"
ANNUAL_LOSS_PATH = NOTEBOOK10_OUTPUT_DIR / "paired_annual_loss_series.csv.gz"
PHASE1_REINSURANCE_FORMULA_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_6_insurance_terms"
    / "notebook_6_cell_6_reinsurance_formula_specification.json"
)
PHASE1_REINSURANCE_SUMMARY_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "notebook_6_insurance_terms"
    / "notebook_6_cell_7_summary.json"
)
MODULE_PATH = PROJECT_ROOT / "tools" / "reinsurance_capital.py"

EVENT_REINSURANCE_PATH = OUTPUT_DIR / "paired_reinsurance_event_loss.csv.gz"
ANNUAL_REINSURANCE_PATH = OUTPUT_DIR / "paired_reinsurance_annual_loss.csv.gz"
INPUT_VALIDATION_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_input_validation.csv"
CONTROL_VALIDATION_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_control_validation.csv"
MODEL_SPECIFICATION_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_model_specification.json"
PROGRAM_SUMMARY_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_program_summary.csv"
OCCURRENCE_GRID_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_occurrence_design_grid.csv"
AGGREGATE_GRID_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_aggregate_design_grid.csv"
PML_TABLE_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_pml_table.csv"
REQUIRED_LIMIT_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_required_limit_summary.csv"
DIVERSIFICATION_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_diversification_summary.csv"
UNCERTAINTY_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_uncertainty_summary.csv"
BREAK_EVEN_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_break_even_premium_grid.csv"
RAROC_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_raroc_assumption_grid.csv"
FINAL_VALIDATION_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_final_validation.csv"
FINAL_HANDOFF_PATH = NOTEBOOK11_METADATA_DIR / "notebook_11_final_handoff.json"


## Financial and tail-risk definitions

For occurrence loss $X_e$, attachment $A$, limit $L$, and participation
$p$, the occurrence recovery is

$$
C_e=p\min\left[\max(X_e-A,0),L\right],
\qquad N_e=X_e-C_e.
$$

For an annual aggregate subject loss $N_y$, the aggregate recovery is

$$
C_{\mathrm{agg},y}
=\min\left[\max(N_y-A_{\mathrm{agg}},0),L_{\mathrm{agg}}\right].
$$

The stacked program applies the occurrence layer first and the annual
aggregate layer second. Every AAL uses all 2,000,000 catalog years,
including zero-event years.

For confidence $q$, this notebook uses a fixed-count empirical tail of
$k=\lceil N(1-q)\rceil$ annual observations. VaR is the smallest loss in
that tail and TVaR is the mean of exactly those $k$ observations. This
avoids ambiguity from ties at zero.

The catalog is sparse enough that 99.0% and 99.5% VaR may be zero. The
signed quantity $\mathrm{VaR}_{99.5}-\mathrm{AAL}$ is retained exactly as
specified, but it is flagged as non-informative when VaR is zero. The
supplemental TVaR tail capital is therefore essential for decisions.

Portfolio diversification is reported as
`diversification_benefit = 1 - portfolio_risk / sum_standalone_risk`
for AAL, VaR, and TVaR, using the 470 buildings as standalone units.


In [ ]:
input_validation_rows: list[dict[str, object]] = []
required_paths = [
    NOTEBOOK10_HANDOFF_PATH,
    NOTEBOOK10_VALIDATION_PATH,
    NOTEBOOK10_PML_PATH,
    BUILDING_LOSS_PATH,
    EVENT_LOSS_PATH,
    ANNUAL_LOSS_PATH,
    PHASE1_REINSURANCE_FORMULA_PATH,
    PHASE1_REINSURANCE_SUMMARY_PATH,
    MODULE_PATH,
]
for path in required_paths:
    add_check(
        input_validation_rows,
        f"required_input:{project_relative_path(path)}",
        path.is_file() and path.stat().st_size > 0,
        f"path={project_relative_path(path)}; exists={path.is_file()}",
    )
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    write_csv(pd.DataFrame(input_validation_rows), INPUT_VALIDATION_PATH)
    raise FileNotFoundError(f"Notebook 11 inputs are missing: {missing_paths}")

notebook10_handoff = load_json(NOTEBOOK10_HANDOFF_PATH)
notebook10_validation = pd.read_csv(NOTEBOOK10_VALIDATION_PATH)
notebook10_pml = pd.read_csv(NOTEBOOK10_PML_PATH)
phase1_formula = load_json(PHASE1_REINSURANCE_FORMULA_PATH)
phase1_summary = load_json(PHASE1_REINSURANCE_SUMMARY_PATH)

add_check(
    input_validation_rows,
    "notebook10_handoff_hash",
    sha256_file(NOTEBOOK10_HANDOFF_PATH) == EXPECTED_HASHES["notebook10_handoff"],
    f"sha256={sha256_file(NOTEBOOK10_HANDOFF_PATH)}",
)
add_check(
    input_validation_rows,
    "notebook10_validation_hash",
    sha256_file(NOTEBOOK10_VALIDATION_PATH)
    == EXPECTED_HASHES["notebook10_validation"],
    f"sha256={sha256_file(NOTEBOOK10_VALIDATION_PATH)}",
)
add_check(
    input_validation_rows,
    "phase1_reinsurance_formula_hash",
    sha256_lf_normalized_text(PHASE1_REINSURANCE_FORMULA_PATH)
    == EXPECTED_HASHES["phase1_reinsurance_formula"],
    (
        "lf_normalized_sha256="
        f"{sha256_lf_normalized_text(PHASE1_REINSURANCE_FORMULA_PATH)}; "
        f"raw_sha256={sha256_file(PHASE1_REINSURANCE_FORMULA_PATH)}"
    ),
)
add_check(
    input_validation_rows,
    "notebook10_complete",
    notebook10_handoff.get("notebook10_complete") is True
    and notebook10_handoff.get("schema_version")
    == "notebook10_correlated_damage_loss_handoff_v1",
    (
        f"complete={notebook10_handoff.get('notebook10_complete')}; "
        f"schema={notebook10_handoff.get('schema_version')}"
    ),
)
add_check(
    input_validation_rows,
    "notebook10_dimensions",
    notebook10_handoff["catalog"]["annual_rows"] == EXPECTED_CATALOG_YEARS
    and notebook10_handoff["frozen_controls"]["occurrences"]
    == EXPECTED_OCCURRENCES
    and notebook10_handoff["frozen_controls"]["sites"] == EXPECTED_SITES,
    (
        f"annual_rows={notebook10_handoff['catalog']['annual_rows']}; "
        f"occurrences={notebook10_handoff['frozen_controls']['occurrences']}; "
        f"sites={notebook10_handoff['frozen_controls']['sites']}"
    ),
)
notebook10_critical_failures = notebook10_validation.loc[
    notebook10_validation["severity"].astype(str).str.lower().eq("critical")
    & ~normalize_boolean_series(notebook10_validation["passed"])
]
add_check(
    input_validation_rows,
    "notebook10_validation_passed",
    notebook10_critical_failures.empty,
    (
        f"checks={len(notebook10_validation)}; "
        f"critical_failures={len(notebook10_critical_failures)}"
    ),
)

inventory_failures: list[str] = []
for item in notebook10_handoff["artifact_inventory"]:
    path = resolve_repository_path(item["path"])
    if not path.is_file():
        inventory_failures.append(f"missing:{item['path']}")
        continue
    if int(path.stat().st_size) != int(item["bytes"]):
        inventory_failures.append(f"bytes:{item['path']}")
        continue
    if VERIFY_INPUT_HASHES and sha256_file(path) != item["sha256"]:
        inventory_failures.append(f"sha256:{item['path']}")
add_check(
    input_validation_rows,
    "notebook10_artifact_inventory",
    not inventory_failures,
    f"artifacts={len(notebook10_handoff['artifact_inventory'])}; failures={inventory_failures}",
)

layer_selection = phase1_formula["layer_selection"]
frozen_term_errors = {
    key: float(layer_selection[source_key]) - float(FROZEN_LAYER[key])
    for key, source_key in [
        ("attachment_2022_usd", "attachment_2022_usd"),
        ("limit_2022_usd", "limit_2022_usd"),
        ("exhaustion_2022_usd", "exhaustion_2022_usd"),
        ("participation", "reinsurer_participation"),
    ]
}
add_check(
    input_validation_rows,
    "phase1_occurrence_layer_terms_frozen",
    max(abs(value) for value in frozen_term_errors.values())
    <= AGGREGATE_TOLERANCE_USD
    and phase1_formula["layer_scenario_id"]
    == FROZEN_LAYER["layer_scenario_id"],
    f"errors={frozen_term_errors}",
)
add_check(
    input_validation_rows,
    "phase1_exclusions_preserved",
    phase1_formula["annual_aggregate_limit"] is None
    and "reinsurance premium" in phase1_formula["excluded_from_baseline"]
    and "annual aggregate limit" in phase1_formula["excluded_from_baseline"],
    "The Phase 1 occurrence benchmark remains unchanged; aggregate and pricing are sensitivities.",
)

event_required_columns = [
    "catalog_year",
    "catalog_event_id",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
    *[
        f"{prefix}_gross_insured_loss_2022_usd"
        for prefix in CASE_PREFIXES.values()
    ],
]
annual_required_columns = [
    "catalog_year",
    "catalog_occurrence_count",
    *[
        f"{prefix}_gross_insured_{curve}_2022_usd"
        for prefix in CASE_PREFIXES.values()
        for curve in ("aep", "oep")
    ],
]
event_header = pd.read_csv(EVENT_LOSS_PATH, nrows=0)
annual_header = pd.read_csv(ANNUAL_LOSS_PATH, nrows=0)
add_check(
    input_validation_rows,
    "event_loss_schema",
    set(event_required_columns).issubset(event_header.columns),
    f"required={len(event_required_columns)}; actual={len(event_header.columns)}",
)
add_check(
    input_validation_rows,
    "annual_loss_schema",
    set(annual_required_columns).issubset(annual_header.columns),
    f"required={len(annual_required_columns)}; actual={len(annual_header.columns)}",
)

event_summary = pd.read_csv(EVENT_LOSS_PATH, usecols=event_required_columns)
annual = pd.read_csv(ANNUAL_LOSS_PATH, usecols=annual_required_columns)
add_check(
    input_validation_rows,
    "event_dimensions",
    len(event_summary) == EXPECTED_OCCURRENCES
    and event_summary["occurrence_id"].nunique() == EXPECTED_OCCURRENCES,
    f"rows={len(event_summary)}; occurrences={event_summary['occurrence_id'].nunique()}",
)
add_check(
    input_validation_rows,
    "annual_dimensions_and_zero_years",
    len(annual) == EXPECTED_CATALOG_YEARS
    and annual["catalog_year"].iloc[0] == 1
    and annual["catalog_year"].iloc[-1] == EXPECTED_CATALOG_YEARS
    and int(np.count_nonzero(annual["catalog_occurrence_count"]))
    == EXPECTED_OCCUPIED_YEARS
    and int(np.count_nonzero(annual["catalog_occurrence_count"].eq(0)))
    == EXPECTED_ZERO_EVENT_YEARS,
    (
        f"rows={len(annual)}; occupied="
        f"{int(np.count_nonzero(annual['catalog_occurrence_count']))}; "
        f"zero_event={int(np.count_nonzero(annual['catalog_occurrence_count'].eq(0)))}"
    ),
)

model_specification = {
    "pipeline_version": PIPELINE_VERSION,
    "dependence_cases": list(CASE_PREFIXES),
    "currency": "USD",
    "dollar_year": 2022,
    "declared_catalog_years": EXPECTED_CATALOG_YEARS,
    "frozen_occurrence_layer": FROZEN_LAYER,
    "fixed_programs": [
        "NO_REINSURANCE",
        "FROZEN_OCCURRENCE_XOL",
        "STANDALONE_AGGREGATE",
        "STACKED_OCCURRENCE_PLUS_AGGREGATE",
    ],
    "stacking_order": (
        "Apply the frozen occurrence XoL to each event, sum occurrence-retained "
        "loss by year, then apply the annual aggregate stop-loss."
    ),
    "aggregate_benchmark_terms": (
        "The benchmark aggregate layer uses the same dollar attachment, limit, "
        "and participation as the frozen occurrence layer."
    ),
    "tail_estimator": (
        "For q, select exactly ceil(N*(1-q)) largest annual losses; VaR is "
        "the smallest selected loss and TVaR is their mean."
    ),
    "economic_capital": {
        "specified_primary": "retained AEP VaR(99.5%) minus retained AAL",
        "floored_diagnostic": "max(specified primary, 0)",
        "supplemental_tail": "retained AEP TVaR(99.5%) minus retained AAL",
        "sparse_catalog_note": (
            "VaR-based capital is flagged non-informative when VaR is zero."
        ),
    },
    "occurrence_grid": {
        "attachment_return_periods_years": ATTACHMENT_RETURN_PERIODS,
        "exhaustion_return_periods_years": EXHAUSTION_RETURN_PERIODS,
        "calibration_case": "I0_PHASE1_INDEPENDENT",
        "calibration_curve": "gross insured OEP PML",
    },
    "aggregate_grid": {
        "attachment_return_periods_years": ATTACHMENT_RETURN_PERIODS,
        "exhaustion_return_periods_years": EXHAUSTION_RETURN_PERIODS,
        "calibration_case": "I0_PHASE1_INDEPENDENT",
        "calibration_curve": "gross insured AEP PML",
    },
    "raroc": {
        "equation": "(earned premium - expenses - retained AAL - reinsurance premium) / economic capital",
        "premium_multiples_of_gross_aal": PREMIUM_MULTIPLES_OF_GROSS_AAL,
        "expense_ratios": EXPENSE_RATIOS,
        "ceded_price_multipliers": CEDED_PRICE_MULTIPLIERS,
        "interpretation": "scenario grid only; no central price or recommended quote",
    },
    "bootstrap": {
        "replicates": BOOTSTRAP_REPLICATES,
        "seed": BOOTSTRAP_SEED,
        "method": "paired multinomial resampling of catalog years",
    },
    "module_sha256": sha256_file(MODULE_PATH),
    "notebook10_handoff_sha256": sha256_file(NOTEBOOK10_HANDOFF_PATH),
    "phase1_reinsurance_formula_sha256": sha256_lf_normalized_text(
        PHASE1_REINSURANCE_FORMULA_PATH
    ),
}
write_json(MODEL_SPECIFICATION_PATH, model_specification)
input_validation = pd.DataFrame(input_validation_rows)
write_csv(input_validation, INPUT_VALIDATION_PATH)
input_failures = input_validation.loc[
    input_validation["severity"].eq("critical")
    & ~normalize_boolean_series(input_validation["passed"])
]
if not input_failures.empty:
    display(input_failures)
    raise RuntimeError("Notebook 11 input validation failed.")

print("=" * 78)
print("NOTEBOOK 11 CELL 1: FROZEN REINSURANCE INPUTS VALIDATED")
print("=" * 78)
print(f"Dependence cases:             {len(CASE_PREFIXES):,}")
print(f"Catalog occurrences:          {len(event_summary):,}")
print(f"Annual rows including zeros:  {len(annual):,}")
print(f"Input validation checks:      {len(input_validation):,}")
print(f"Frozen attachment:            ${FROZEN_LAYER['attachment_2022_usd']:,.2f}")
print(f"Frozen limit:                 ${FROZEN_LAYER['limit_2022_usd']:,.2f}")
print("Next: exact I0 occurrence-XoL reconstruction against Phase 1.")


In [ ]:
control_validation_rows: list[dict[str, object]] = []
i0_gross = event_summary["i0_gross_insured_loss_2022_usd"].to_numpy(
    dtype=np.float64
)
i0_layer = apply_occurrence_xol(
    i0_gross,
    FROZEN_LAYER["attachment_2022_usd"],
    FROZEN_LAYER["limit_2022_usd"],
    FROZEN_LAYER["participation"],
    classification_tolerance=ROW_TOLERANCE_USD,
)
i0_ceded = np.asarray(i0_layer["ceded_loss_2022_usd"], dtype=np.float64)
i0_retained = np.asarray(i0_layer["retained_loss_2022_usd"], dtype=np.float64)
i0_annualized = annualize_occurrence_program(
    event_summary["catalog_year"].to_numpy(dtype=np.int64),
    i0_gross,
    i0_ceded,
    EXPECTED_CATALOG_YEARS,
)
control_observed = {
    "triggering_occurrences": int(np.count_nonzero(i0_layer["triggered"])),
    "exhausting_occurrences": int(np.count_nonzero(i0_layer["exhausted"])),
    "ceded_loss_total_2022_usd": float(i0_ceded.sum(dtype=np.float64)),
    "retained_loss_total_2022_usd": float(i0_retained.sum(dtype=np.float64)),
    "ceded_aal_2022_usd": float(
        np.asarray(i0_annualized["ceded_aep_2022_usd"]).mean()
    ),
    "retained_aal_2022_usd": float(
        np.asarray(i0_annualized["retained_aep_2022_usd"]).mean()
    ),
    "maximum_ceded_occurrence_loss_2022_usd": float(i0_ceded.max()),
    "maximum_retained_occurrence_loss_2022_usd": float(i0_retained.max()),
    "maximum_ceded_aep_2022_usd": float(
        np.asarray(i0_annualized["ceded_aep_2022_usd"]).max()
    ),
}
for metric, expected in EXPECTED_PHASE1_REINSURANCE.items():
    observed = control_observed[metric]
    tolerance = 0 if metric.endswith("occurrences") else AGGREGATE_TOLERANCE_USD
    add_check(
        control_validation_rows,
        f"phase1_control:{metric}",
        abs(float(observed) - float(expected)) <= tolerance,
        f"observed={observed}; expected={expected}; tolerance={tolerance}",
    )

annual_gross_error = float(
    np.max(
        np.abs(
            np.asarray(i0_annualized["gross_aep_2022_usd"])
            - annual["i0_gross_insured_aep_2022_usd"].to_numpy(float)
        )
    )
)
annual_oep_error = float(
    np.max(
        np.abs(
            np.asarray(i0_annualized["gross_oep_2022_usd"])
            - annual["i0_gross_insured_oep_2022_usd"].to_numpy(float)
        )
    )
)
add_check(
    control_validation_rows,
    "i0_annual_gross_reconstruction",
    max(annual_gross_error, annual_oep_error) <= AGGREGATE_TOLERANCE_USD,
    f"maximum_aep_error={annual_gross_error:.6e}; maximum_oep_error={annual_oep_error:.6e}",
)
add_check(
    control_validation_rows,
    "i0_occurrence_conservation",
    float(np.max(np.abs(i0_gross - i0_ceded - i0_retained)))
    <= ROW_TOLERANCE_USD,
    (
        "maximum_error="
        f"{float(np.max(np.abs(i0_gross-i0_ceded-i0_retained))):.6e}"
    ),
)
add_check(
    control_validation_rows,
    "frozen_layer_bounds",
    i0_ceded.min(initial=0.0) >= 0.0
    and i0_ceded.max(initial=0.0)
    <= FROZEN_LAYER["limit_2022_usd"] + ROW_TOLERANCE_USD,
    f"ceded_range={i0_ceded.min(initial=0.0):.6f}-{i0_ceded.max(initial=0.0):.6f}",
)

control_validation = pd.DataFrame(control_validation_rows)
write_csv(control_validation, CONTROL_VALIDATION_PATH)
control_failures = control_validation.loc[
    control_validation["severity"].eq("critical")
    & ~normalize_boolean_series(control_validation["passed"])
]
if not control_failures.empty:
    display(control_failures)
    raise RuntimeError("Notebook 11 exact I0 reinsurance control failed.")

print("=" * 78)
print("NOTEBOOK 11 CELL 2: EXACT I0 REINSURANCE CONTROL PASSED")
print("=" * 78)
print(f"Triggering occurrences:       {control_observed['triggering_occurrences']:,}")
print(f"Exhausting occurrences:       {control_observed['exhausting_occurrences']:,}")
print(f"Ceded AAL:                    ${control_observed['ceded_aal_2022_usd']:,.2f}")
print(f"Retained AAL:                 ${control_observed['retained_aal_2022_usd']:,.2f}")
print(f"Validation checks:            {len(control_validation):,}")
print("Next: apply the four fixed programs to I0, C1, and C2.")


In [ ]:
event_identifiers = [
    "catalog_year",
    "catalog_event_id",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_id",
    "source_type",
    "magnitude",
]
event_reinsurance = event_summary[event_identifiers].copy()
annual_reinsurance = annual[["catalog_year", "catalog_occurrence_count"]].copy()
fixed_program_arrays: dict[tuple[str, str], dict[str, np.ndarray]] = {}
program_rows: list[dict[str, object]] = []
pml_rows: list[dict[str, object]] = []

for case_name, prefix in CASE_PREFIXES.items():
    gross_event = event_summary[
        f"{prefix}_gross_insured_loss_2022_usd"
    ].to_numpy(dtype=np.float64)
    gross_aep = annual[
        f"{prefix}_gross_insured_aep_2022_usd"
    ].to_numpy(dtype=np.float64)
    gross_oep = annual[
        f"{prefix}_gross_insured_oep_2022_usd"
    ].to_numpy(dtype=np.float64)

    occurrence_layer = apply_occurrence_xol(
        gross_event,
        FROZEN_LAYER["attachment_2022_usd"],
        FROZEN_LAYER["limit_2022_usd"],
        FROZEN_LAYER["participation"],
        classification_tolerance=ROW_TOLERANCE_USD,
    )
    occurrence_ceded_event = np.asarray(
        occurrence_layer["ceded_loss_2022_usd"], dtype=np.float64
    )
    occurrence_retained_event = np.asarray(
        occurrence_layer["retained_loss_2022_usd"], dtype=np.float64
    )
    occurrence_annual = annualize_occurrence_program(
        event_summary["catalog_year"].to_numpy(dtype=np.int64),
        gross_event,
        occurrence_ceded_event,
        EXPECTED_CATALOG_YEARS,
    )
    occurrence_ceded_aep_unreconciled = np.asarray(
        occurrence_annual["ceded_aep_2022_usd"], dtype=np.float64
    )
    occurrence_retained_aep_unreconciled = np.asarray(
        occurrence_annual["retained_aep_2022_usd"], dtype=np.float64
    )
    occurrence_waterfall = reconcile_annual_waterfall(
        gross_aep,
        occurrence_ceded_aep_unreconciled,
        occurrence_retained_aep_unreconciled,
        tolerance=ROW_TOLERANCE_USD,
    )
    occurrence_ceded_aep = occurrence_waterfall["ceded_aep_2022_usd"]
    occurrence_retained_aep = occurrence_waterfall[
        "retained_aep_2022_usd"
    ]

    standalone = apply_aggregate_stop_loss(
        gross_aep,
        FROZEN_LAYER["attachment_2022_usd"],
        FROZEN_LAYER["limit_2022_usd"],
        FROZEN_LAYER["participation"],
        classification_tolerance=ROW_TOLERANCE_USD,
    )
    standalone_ceded_unreconciled = np.asarray(
        standalone["ceded_loss_2022_usd"], dtype=np.float64
    )
    standalone_retained_unreconciled = np.asarray(
        standalone["retained_loss_2022_usd"], dtype=np.float64
    )
    standalone_waterfall = reconcile_annual_waterfall(
        gross_aep,
        standalone_ceded_unreconciled,
        standalone_retained_unreconciled,
        tolerance=ROW_TOLERANCE_USD,
    )
    standalone_ceded = standalone_waterfall["ceded_aep_2022_usd"]
    standalone_retained = standalone_waterfall[
        "retained_aep_2022_usd"
    ]

    stacked_aggregate = apply_aggregate_stop_loss(
        occurrence_retained_aep,
        FROZEN_LAYER["attachment_2022_usd"],
        FROZEN_LAYER["limit_2022_usd"],
        FROZEN_LAYER["participation"],
        classification_tolerance=ROW_TOLERANCE_USD,
    )
    stacked_aggregate_ceded = np.asarray(
        stacked_aggregate["ceded_loss_2022_usd"], dtype=np.float64
    )
    stacked_final_retained_unreconciled = np.asarray(
        stacked_aggregate["retained_loss_2022_usd"], dtype=np.float64
    )
    stacked_total_ceded_unreconciled = (
        occurrence_ceded_aep + stacked_aggregate_ceded
    )
    stacked_waterfall = reconcile_annual_waterfall(
        gross_aep,
        stacked_total_ceded_unreconciled,
        stacked_final_retained_unreconciled,
        tolerance=ROW_TOLERANCE_USD,
    )
    stacked_total_ceded = stacked_waterfall["ceded_aep_2022_usd"]
    stacked_final_retained = stacked_waterfall[
        "retained_aep_2022_usd"
    ]

    event_reinsurance[f"{prefix}_gross_insured_loss_2022_usd"] = gross_event
    event_reinsurance[
        f"{prefix}_frozen_occurrence_ceded_loss_2022_usd"
    ] = occurrence_ceded_event
    event_reinsurance[
        f"{prefix}_frozen_occurrence_retained_loss_2022_usd"
    ] = occurrence_retained_event

    annual_reinsurance[f"{prefix}_gross_insured_aep_2022_usd"] = gross_aep
    annual_reinsurance[f"{prefix}_gross_insured_oep_2022_usd"] = gross_oep
    annual_reinsurance[
        f"{prefix}_frozen_occurrence_ceded_aep_2022_usd"
    ] = occurrence_ceded_aep
    annual_reinsurance[
        f"{prefix}_frozen_occurrence_ceded_oep_2022_usd"
    ] = np.asarray(occurrence_annual["ceded_oep_2022_usd"])
    annual_reinsurance[
        f"{prefix}_frozen_occurrence_retained_aep_2022_usd"
    ] = occurrence_retained_aep
    annual_reinsurance[
        f"{prefix}_frozen_occurrence_retained_oep_2022_usd"
    ] = np.asarray(occurrence_annual["retained_oep_2022_usd"])
    annual_reinsurance[
        f"{prefix}_standalone_aggregate_ceded_aep_2022_usd"
    ] = standalone_ceded
    annual_reinsurance[
        f"{prefix}_standalone_aggregate_retained_aep_2022_usd"
    ] = standalone_retained
    annual_reinsurance[
        f"{prefix}_stacked_aggregate_ceded_aep_2022_usd"
    ] = stacked_aggregate_ceded
    annual_reinsurance[
        f"{prefix}_stacked_total_ceded_aep_2022_usd"
    ] = stacked_total_ceded
    annual_reinsurance[
        f"{prefix}_stacked_final_retained_aep_2022_usd"
    ] = stacked_final_retained

    fixed_program_arrays[(case_name, "NO_REINSURANCE")] = {
        "gross_aep": gross_aep,
        "ceded_aep": np.zeros_like(gross_aep),
        "retained_aep": gross_aep.copy(),
        "gross_oep": gross_oep,
        "ceded_oep": np.zeros_like(gross_oep),
        "retained_oep": gross_oep.copy(),
    }
    fixed_program_arrays[(case_name, "FROZEN_OCCURRENCE_XOL")] = {
        "gross_aep": gross_aep,
        "ceded_aep": occurrence_ceded_aep,
        "retained_aep": occurrence_retained_aep,
        "gross_oep": gross_oep,
        "ceded_oep": np.asarray(occurrence_annual["ceded_oep_2022_usd"]),
        "retained_oep": np.asarray(
            occurrence_annual["retained_oep_2022_usd"]
        ),
    }
    fixed_program_arrays[(case_name, "STANDALONE_AGGREGATE")] = {
        "gross_aep": gross_aep,
        "ceded_aep": standalone_ceded,
        "retained_aep": standalone_retained,
    }
    fixed_program_arrays[
        (case_name, "STACKED_OCCURRENCE_PLUS_AGGREGATE")
    ] = {
        "gross_aep": gross_aep,
        "ceded_aep": stacked_total_ceded,
        "retained_aep": stacked_final_retained,
    }

    for program in [
        "NO_REINSURANCE",
        "FROZEN_OCCURRENCE_XOL",
        "STANDALONE_AGGREGATE",
        "STACKED_OCCURRENCE_PLUS_AGGREGATE",
    ]:
        arrays = fixed_program_arrays[(case_name, program)]
        metrics = annual_program_metrics(
            arrays["gross_aep"],
            arrays["ceded_aep"],
            arrays["retained_aep"],
        )
        retained_2500, retained_2500_rank = empirical_pml(
            arrays["retained_aep"], 2_500
        )
        gross_2500, _ = empirical_pml(arrays["gross_aep"], 2_500)
        metrics.update(
            {
                "case_name": case_name,
                "case_prefix": prefix,
                "program": program,
                "occurrence_attachment_2022_usd": (
                    FROZEN_LAYER["attachment_2022_usd"]
                    if program
                    in {
                        "FROZEN_OCCURRENCE_XOL",
                        "STACKED_OCCURRENCE_PLUS_AGGREGATE",
                    }
                    else 0.0
                ),
                "occurrence_limit_2022_usd": (
                    FROZEN_LAYER["limit_2022_usd"]
                    if program
                    in {
                        "FROZEN_OCCURRENCE_XOL",
                        "STACKED_OCCURRENCE_PLUS_AGGREGATE",
                    }
                    else 0.0
                ),
                "aggregate_attachment_2022_usd": (
                    FROZEN_LAYER["attachment_2022_usd"]
                    if program
                    in {
                        "STANDALONE_AGGREGATE",
                        "STACKED_OCCURRENCE_PLUS_AGGREGATE",
                    }
                    else 0.0
                ),
                "aggregate_limit_2022_usd": (
                    FROZEN_LAYER["limit_2022_usd"]
                    if program
                    in {
                        "STANDALONE_AGGREGATE",
                        "STACKED_OCCURRENCE_PLUS_AGGREGATE",
                    }
                    else 0.0
                ),
                "gross_aep_2500yr_pml_2022_usd": gross_2500,
                "retained_aep_2500yr_pml_2022_usd": retained_2500,
                "retained_aep_2500yr_order_statistic_rank": retained_2500_rank,
            }
        )
        ceded_aal = float(metrics["ceded_aal_2022_usd"])
        metrics[
            "tvar99_5_capital_relief_per_expected_ceded_dollar"
        ] = (
            float(metrics["tvar_capital_relief_99_5_2022_usd"])
            / ceded_aal
            if ceded_aal > 0.0
            else np.nan
        )
        metrics[
            "pml2500_relief_per_expected_ceded_dollar"
        ] = (
            (gross_2500 - retained_2500) / ceded_aal
            if ceded_aal > 0.0
            else np.nan
        )
        program_rows.append(metrics)

        curves = ["aep"]
        if "gross_oep" in arrays:
            curves.append("oep")
        for curve in curves:
            for loss_basis in ["gross", "ceded", "retained"]:
                values = np.asarray(arrays[f"{loss_basis}_{curve}"])
                for return_period in RETURN_PERIODS:
                    pml, rank = empirical_pml(values, return_period)
                    pml_rows.append(
                        {
                            "case_name": case_name,
                            "case_prefix": prefix,
                            "program": program,
                            "loss_basis": loss_basis,
                            "curve_type": curve,
                            "return_period_years": return_period,
                            "order_statistic_rank": rank,
                            "tail_support_sufficient": rank >= 20,
                            "pml_2022_usd": pml,
                        }
                    )

program_summary = pd.DataFrame(program_rows)
pml_table = pd.DataFrame(pml_rows)
write_gzip_csv_deterministic(event_reinsurance, EVENT_REINSURANCE_PATH)
write_gzip_csv_deterministic(annual_reinsurance, ANNUAL_REINSURANCE_PATH)
write_csv(program_summary, PROGRAM_SUMMARY_PATH)
write_csv(pml_table, PML_TABLE_PATH)

display(
    program_summary[
        [
            "case_name",
            "program",
            "ceded_aal_2022_usd",
            "retained_aal_2022_usd",
            "retained_tvar_99_5_2022_usd",
            "retained_aep_2500yr_pml_2022_usd",
        ]
    ]
)
print("=" * 78)
print("NOTEBOOK 11 CELL 3: FIXED REINSURANCE PROGRAMS COMPLETE")
print("=" * 78)
print(f"Programs per case:            4")
print(f"Program summary rows:         {len(program_summary):,}")
print(f"PML rows:                     {len(pml_table):,}")
print(f"Event output rows:            {len(event_reinsurance):,}")
print(f"Annual output rows:           {len(annual_reinsurance):,}")
print("Next: evaluate common occurrence and aggregate design grids.")


In [ ]:
def pml_lookup(
    prefix: str,
    curve_type: str,
    return_period: int,
) -> tuple[float, int]:
    row = notebook10_pml.loc[
        notebook10_pml["case_prefix"].eq(prefix)
        & notebook10_pml["loss_basis"].eq("gross_insured")
        & notebook10_pml["curve_type"].eq(curve_type)
        & notebook10_pml["return_period_years"].eq(return_period)
    ]
    if len(row) != 1:
        raise RuntimeError("Notebook 10 PML lookup is not unique.")
    return float(row.iloc[0]["pml_2022_usd"]), int(
        row.iloc[0]["order_statistic_rank"]
    )


occurrence_terms: list[dict[str, object]] = []
aggregate_terms: list[dict[str, object]] = []
for attachment_rp in ATTACHMENT_RETURN_PERIODS:
    for exhaustion_rp in EXHAUSTION_RETURN_PERIODS:
        occurrence_attachment, occurrence_attachment_rank = pml_lookup(
            "i0", "oep", attachment_rp
        )
        occurrence_exhaustion, occurrence_exhaustion_rank = pml_lookup(
            "i0", "oep", exhaustion_rp
        )
        occurrence_limit = occurrence_exhaustion - occurrence_attachment
        if occurrence_limit > ROW_TOLERANCE_USD:
            occurrence_terms.append(
                {
                    "attachment_return_period_years": attachment_rp,
                    "exhaustion_return_period_years": exhaustion_rp,
                    "attachment_order_statistic_rank": occurrence_attachment_rank,
                    "exhaustion_order_statistic_rank": occurrence_exhaustion_rank,
                    "attachment_2022_usd": occurrence_attachment,
                    "limit_2022_usd": occurrence_limit,
                    "exhaustion_2022_usd": occurrence_exhaustion,
                }
            )
        aggregate_attachment, aggregate_attachment_rank = pml_lookup(
            "i0", "aep", attachment_rp
        )
        aggregate_exhaustion, aggregate_exhaustion_rank = pml_lookup(
            "i0", "aep", exhaustion_rp
        )
        aggregate_limit = aggregate_exhaustion - aggregate_attachment
        if aggregate_limit > ROW_TOLERANCE_USD:
            aggregate_terms.append(
                {
                    "attachment_return_period_years": attachment_rp,
                    "exhaustion_return_period_years": exhaustion_rp,
                    "attachment_order_statistic_rank": aggregate_attachment_rank,
                    "exhaustion_order_statistic_rank": aggregate_exhaustion_rank,
                    "attachment_2022_usd": aggregate_attachment,
                    "limit_2022_usd": aggregate_limit,
                    "exhaustion_2022_usd": aggregate_exhaustion,
                }
            )

occurrence_grid_rows: list[dict[str, object]] = []
aggregate_grid_rows: list[dict[str, object]] = []
required_limit_rows: list[dict[str, object]] = []

for case_name, prefix in CASE_PREFIXES.items():
    gross_event = event_summary[
        f"{prefix}_gross_insured_loss_2022_usd"
    ].to_numpy(dtype=np.float64)
    gross_aep = annual[
        f"{prefix}_gross_insured_aep_2022_usd"
    ].to_numpy(dtype=np.float64)
    frozen_occurrence_retained = annual_reinsurance[
        f"{prefix}_frozen_occurrence_retained_aep_2022_usd"
    ].to_numpy(dtype=np.float64)
    frozen_occurrence_ceded = annual_reinsurance[
        f"{prefix}_frozen_occurrence_ceded_aep_2022_usd"
    ].to_numpy(dtype=np.float64)

    for design_number, terms in enumerate(occurrence_terms, start=1):
        layer = apply_occurrence_xol(
            gross_event,
            float(terms["attachment_2022_usd"]),
            float(terms["limit_2022_usd"]),
            1.0,
            classification_tolerance=ROW_TOLERANCE_USD,
        )
        ceded_event = np.asarray(layer["ceded_loss_2022_usd"])
        annualized = annualize_occurrence_program(
            event_summary["catalog_year"].to_numpy(dtype=np.int64),
            gross_event,
            ceded_event,
            EXPECTED_CATALOG_YEARS,
        )
        ceded_aep_unreconciled = np.asarray(
            annualized["ceded_aep_2022_usd"]
        )
        retained_aep_unreconciled = np.asarray(
            annualized["retained_aep_2022_usd"]
        )
        occurrence_waterfall = reconcile_annual_waterfall(
            gross_aep,
            ceded_aep_unreconciled,
            retained_aep_unreconciled,
            tolerance=ROW_TOLERANCE_USD,
        )
        ceded_aep = occurrence_waterfall["ceded_aep_2022_usd"]
        retained_aep = occurrence_waterfall[
            "retained_aep_2022_usd"
        ]
        metrics = annual_program_metrics(gross_aep, ceded_aep, retained_aep)
        retained_2500, _ = empirical_pml(retained_aep, 2_500)
        ceded_aal = float(metrics["ceded_aal_2022_usd"])
        occurrence_grid_rows.append(
            {
                "design_id": f"OCC_{design_number:02d}",
                "case_name": case_name,
                "case_prefix": prefix,
                **terms,
                "participation": 1.0,
                "triggering_occurrences": int(np.count_nonzero(layer["triggered"])),
                "exhausting_occurrences": int(np.count_nonzero(layer["exhausted"])),
                "attachment_annual_frequency": (
                    int(np.count_nonzero(layer["triggered"]))
                    / EXPECTED_CATALOG_YEARS
                ),
                "exhaustion_annual_frequency": (
                    int(np.count_nonzero(layer["exhausted"]))
                    / EXPECTED_CATALOG_YEARS
                ),
                "retained_aep_2500yr_pml_2022_usd": retained_2500,
                "tvar99_5_capital_relief_per_expected_ceded_dollar": (
                    float(metrics["tvar_capital_relief_99_5_2022_usd"])
                    / ceded_aal
                    if ceded_aal > 0.0
                    else np.nan
                ),
                **metrics,
            }
        )

    for design_number, terms in enumerate(aggregate_terms, start=1):
        for structure, subject, prior_ceded in [
            (
                "STANDALONE_ANNUAL_AGGREGATE",
                gross_aep,
                np.zeros_like(gross_aep),
            ),
            (
                "STACKED_AFTER_FROZEN_OCCURRENCE",
                frozen_occurrence_retained,
                frozen_occurrence_ceded,
            ),
        ]:
            aggregate_layer = apply_aggregate_stop_loss(
                subject,
                float(terms["attachment_2022_usd"]),
                float(terms["limit_2022_usd"]),
                1.0,
                classification_tolerance=ROW_TOLERANCE_USD,
            )
            aggregate_ceded = np.asarray(
                aggregate_layer["ceded_loss_2022_usd"]
            )
            final_retained_unreconciled = np.asarray(
                aggregate_layer["retained_loss_2022_usd"],
                dtype=np.float64,
            )
            total_ceded_unreconciled = prior_ceded + aggregate_ceded
            aggregate_waterfall = reconcile_annual_waterfall(
                gross_aep,
                total_ceded_unreconciled,
                final_retained_unreconciled,
                tolerance=ROW_TOLERANCE_USD,
            )
            total_ceded = aggregate_waterfall["ceded_aep_2022_usd"]
            final_retained = aggregate_waterfall[
                "retained_aep_2022_usd"
            ]
            metrics = annual_program_metrics(
                gross_aep, total_ceded, final_retained
            )
            retained_2500, _ = empirical_pml(final_retained, 2_500)
            ceded_aal = float(metrics["ceded_aal_2022_usd"])
            aggregate_grid_rows.append(
                {
                    "design_id": f"AGG_{design_number:02d}",
                    "case_name": case_name,
                    "case_prefix": prefix,
                    "structure": structure,
                    "application_order": (
                        "annual aggregate only"
                        if structure == "STANDALONE_ANNUAL_AGGREGATE"
                        else "frozen occurrence XoL, then annual aggregate"
                    ),
                    "calibration_basis": "I0 gross insured AEP PML",
                    **terms,
                    "participation": 1.0,
                    "aggregate_trigger_years": int(
                        np.count_nonzero(aggregate_layer["triggered"])
                    ),
                    "aggregate_exhaustion_years": int(
                        np.count_nonzero(aggregate_layer["exhausted"])
                    ),
                    "retained_aep_2500yr_pml_2022_usd": retained_2500,
                    "tvar99_5_capital_relief_per_expected_ceded_dollar": (
                        float(metrics["tvar_capital_relief_99_5_2022_usd"])
                        / ceded_aal
                        if ceded_aal > 0.0
                        else np.nan
                    ),
                    **metrics,
                }
            )

i0_frozen = fixed_program_arrays[
    ("I0_PHASE1_INDEPENDENT", "FROZEN_OCCURRENCE_XOL")
]["retained_aep"]
target_pml_2500, _ = empirical_pml(i0_frozen, 2_500)
_, target_tvar_99_5, _ = empirical_var_tvar(i0_frozen, 0.995)
targets = [
    ("aep_pml", 2_500.0, target_pml_2500),
    ("aep_tvar", 0.995, target_tvar_99_5),
]
for case_name, prefix in CASE_PREFIXES.items():
    gross_event = event_summary[
        f"{prefix}_gross_insured_loss_2022_usd"
    ].to_numpy(dtype=np.float64)
    for target_kind, target_parameter, target_value in targets:
        result = minimum_occurrence_limit_for_target(
            event_summary["catalog_year"].to_numpy(dtype=np.int64),
            gross_event,
            declared_years=EXPECTED_CATALOG_YEARS,
            attachment=FROZEN_LAYER["attachment_2022_usd"],
            target_kind=target_kind,
            target_parameter=target_parameter,
            target_metric_2022_usd=target_value,
            limit_tolerance_2022_usd=LIMIT_SEARCH_TOLERANCE_USD,
        )
        required_limit_rows.append(
            {
                "case_name": case_name,
                "case_prefix": prefix,
                "target_basis": "I0 frozen occurrence program",
                **result.to_dict(),
                "frozen_limit_2022_usd": FROZEN_LAYER["limit_2022_usd"],
                "required_limit_change_vs_frozen_2022_usd": (
                    result.required_limit_2022_usd
                    - FROZEN_LAYER["limit_2022_usd"]
                    if result.feasible
                    else np.nan
                ),
                "required_limit_change_vs_frozen_percent": (
                    100.0
                    * (
                        result.required_limit_2022_usd
                        / FROZEN_LAYER["limit_2022_usd"]
                        - 1.0
                    )
                    if result.feasible
                    else np.nan
                ),
            }
        )

occurrence_design_grid = pd.DataFrame(occurrence_grid_rows)
aggregate_design_grid = pd.DataFrame(aggregate_grid_rows)
required_limit_summary = pd.DataFrame(required_limit_rows)
write_csv(occurrence_design_grid, OCCURRENCE_GRID_PATH)
write_csv(aggregate_design_grid, AGGREGATE_GRID_PATH)
write_csv(required_limit_summary, REQUIRED_LIMIT_PATH)

display(
    required_limit_summary[
        [
            "case_name",
            "target_kind",
            "target_parameter",
            "feasible",
            "required_limit_2022_usd",
            "required_limit_change_vs_frozen_percent",
        ]
    ]
)
print("=" * 78)
print("NOTEBOOK 11 CELL 4: REINSURANCE DESIGN SENSITIVITY COMPLETE")
print("=" * 78)
print(f"Occurrence designs:           {len(occurrence_terms):,}")
print(f"Occurrence grid rows:         {len(occurrence_design_grid):,}")
print(f"Aggregate designs:            {len(aggregate_terms):,}")
print(f"Aggregate grid rows:          {len(aggregate_design_grid):,}")
print(f"Required-limit targets:       {len(required_limit_summary):,}")
print("Next: quantify diversification, uncertainty, break-even premium, and RAROC.")


In [ ]:
building_columns = [
    "catalog_year",
    "site_id",
    *[
        f"{prefix}_gross_insured_loss_2022_usd"
        for prefix in CASE_PREFIXES.values()
    ],
]
building_loss = pd.read_csv(BUILDING_LOSS_PATH, usecols=building_columns)
diversification_frames: list[pd.DataFrame] = []
for case_name, prefix in CASE_PREFIXES.items():
    loss_column = f"{prefix}_gross_insured_loss_2022_usd"
    case_diversification = diversification_from_sparse_units(
        building_loss[["catalog_year", "site_id", loss_column]],
        unit_column="site_id",
        year_column="catalog_year",
        loss_column=loss_column,
        portfolio_annual_losses=annual_reinsurance[
            f"{prefix}_gross_insured_aep_2022_usd"
        ].to_numpy(dtype=np.float64),
        declared_years=EXPECTED_CATALOG_YEARS,
    )
    case_diversification.insert(0, "case_prefix", prefix)
    case_diversification.insert(0, "case_name", case_name)
    case_diversification.insert(2, "loss_basis", "gross_insured")
    diversification_frames.append(case_diversification)
diversification_summary = pd.concat(
    diversification_frames, ignore_index=True
)
del building_loss
write_csv(diversification_summary, DIVERSIFICATION_PATH)

bootstrap_series: dict[str, np.ndarray] = {}
bootstrap_comparisons: list[tuple[str, str]] = []
for program in [
    "NO_REINSURANCE",
    "FROZEN_OCCURRENCE_XOL",
    "STANDALONE_AGGREGATE",
    "STACKED_OCCURRENCE_PLUS_AGGREGATE",
]:
    for case_name in CASE_PREFIXES:
        label = f"{program}:{case_name}"
        bootstrap_series[label] = fixed_program_arrays[
            (case_name, program)
        ]["retained_aep"]
    bootstrap_comparisons.extend(
        [
            (
                f"{program}:C1_ALDEA22_SUBDUCTION",
                f"{program}:I0_PHASE1_INDEPENDENT",
            ),
            (
                f"{program}:C2_GODA_ATKINSON09",
                f"{program}:I0_PHASE1_INDEPENDENT",
            ),
        ]
    )
bootstrap_metrics = [
    {"name": "aal", "kind": "mean"},
    {"name": "aep_pml_500yr", "kind": "pml", "return_period": 500},
    {"name": "aep_pml_1000yr", "kind": "pml", "return_period": 1_000},
    {"name": "aep_pml_2500yr", "kind": "pml", "return_period": 2_500},
    {"name": "aep_pml_5000yr", "kind": "pml", "return_period": 5_000},
    {"name": "aep_pml_10000yr", "kind": "pml", "return_period": 10_000},
    {"name": "aep_tvar_99_5", "kind": "tvar", "confidence": 0.995},
]
uncertainty_summary = paired_bootstrap_differences(
    bootstrap_series,
    bootstrap_comparisons,
    bootstrap_metrics,
    replicates=BOOTSTRAP_REPLICATES,
    seed=BOOTSTRAP_SEED,
)
uncertainty_summary.insert(
    0,
    "program",
    uncertainty_summary["case_series"].str.split(":").str[0],
)
write_csv(uncertainty_summary, UNCERTAINTY_PATH)

break_even_rows: list[dict[str, object]] = []
raroc_rows: list[dict[str, object]] = []
for _, summary in program_summary.iterrows():
    program = str(summary["program"])
    ceded_price_multipliers = (
        [0.0] if program == "NO_REINSURANCE" else CEDED_PRICE_MULTIPLIERS
    )
    for ceded_price_multiplier in ceded_price_multipliers:
        reinsurance_premium = (
            ceded_price_multiplier * float(summary["ceded_aal_2022_usd"])
        )
        for expense_ratio in EXPENSE_RATIOS:
            break_even_premium = (
                float(summary["retained_aal_2022_usd"])
                + reinsurance_premium
            ) / (1.0 - expense_ratio)
            break_even_rows.append(
                {
                    "case_name": summary["case_name"],
                    "case_prefix": summary["case_prefix"],
                    "program": program,
                    "ceded_price_multiplier": ceded_price_multiplier,
                    "expense_ratio": expense_ratio,
                    "expected_reinsurance_premium_2022_usd": reinsurance_premium,
                    "break_even_required_earned_premium_2022_usd": break_even_premium,
                    "break_even_premium_multiple_of_gross_aal": (
                        break_even_premium
                        / float(summary["gross_aal_2022_usd"])
                    ),
                    "pricing_status": "assumption grid; not a quoted price",
                }
            )
            for premium_multiple in PREMIUM_MULTIPLES_OF_GROSS_AAL:
                earned_premium = (
                    premium_multiple * float(summary["gross_aal_2022_usd"])
                )
                expenses = expense_ratio * earned_premium
                underwriting_result = (
                    earned_premium
                    - expenses
                    - float(summary["retained_aal_2022_usd"])
                    - reinsurance_premium
                )
                var_capital = float(
                    summary[
                        "retained_var_economic_capital_floored_99_5_2022_usd"
                    ]
                )
                tvar_capital = float(
                    summary["retained_tvar_tail_capital_99_5_2022_usd"]
                )
                raroc_rows.append(
                    {
                        "case_name": summary["case_name"],
                        "case_prefix": summary["case_prefix"],
                        "program": program,
                        "premium_multiple_of_gross_aal": premium_multiple,
                        "expense_ratio": expense_ratio,
                        "ceded_price_multiplier": ceded_price_multiplier,
                        "earned_premium_2022_usd": earned_premium,
                        "expenses_2022_usd": expenses,
                        "retained_aal_2022_usd": summary[
                            "retained_aal_2022_usd"
                        ],
                        "expected_reinsurance_premium_2022_usd": reinsurance_premium,
                        "underwriting_result_2022_usd": underwriting_result,
                        "var99_5_economic_capital_floored_2022_usd": var_capital,
                        "tvar99_5_tail_capital_2022_usd": tvar_capital,
                        "raroc_using_var99_5_capital": (
                            underwriting_result / var_capital
                            if var_capital > 0.0
                            else np.nan
                        ),
                        "raroc_using_tvar99_5_tail_capital": (
                            underwriting_result / tvar_capital
                            if tvar_capital > 0.0
                            else np.nan
                        ),
                        "var_capital_informative": var_capital > 0.0,
                        "pricing_status": "assumption grid; not a central estimate",
                    }
                )

break_even_premium_grid = pd.DataFrame(break_even_rows)
raroc_assumption_grid = pd.DataFrame(raroc_rows)
write_csv(break_even_premium_grid, BREAK_EVEN_PATH)
write_csv(raroc_assumption_grid, RAROC_PATH)

print("=" * 78)
print("NOTEBOOK 11 CELL 5: CAPITAL AND DECISION METRICS COMPLETE")
print("=" * 78)
print(f"Diversification rows:         {len(diversification_summary):,}")
print(f"Bootstrap replicates:         {BOOTSTRAP_REPLICATES:,}")
print(f"Uncertainty rows:             {len(uncertainty_summary):,}")
print(f"Break-even grid rows:         {len(break_even_premium_grid):,}")
print(f"RAROC grid rows:              {len(raroc_assumption_grid):,}")
print("Next: run the full reinsurance and capital validation gates.")


In [ ]:
final_validation_rows: list[dict[str, object]] = []
input_validation = pd.read_csv(INPUT_VALIDATION_PATH)
control_validation = pd.read_csv(CONTROL_VALIDATION_PATH)
add_check(
    final_validation_rows,
    "input_and_control_validation_passed",
    normalize_boolean_series(input_validation["passed"]).all()
    and normalize_boolean_series(control_validation["passed"]).all(),
    f"input_checks={len(input_validation)}; control_checks={len(control_validation)}",
)
add_check(
    final_validation_rows,
    "fixed_program_dimensions",
    len(program_summary) == len(CASE_PREFIXES) * 4
    and program_summary["case_name"].nunique() == len(CASE_PREFIXES)
    and program_summary["program"].nunique() == 4,
    (
        f"rows={len(program_summary)}; cases={program_summary['case_name'].nunique()}; "
        f"programs={program_summary['program'].nunique()}"
    ),
)
add_check(
    final_validation_rows,
    "event_and_annual_output_dimensions",
    len(event_reinsurance) == EXPECTED_OCCURRENCES
    and len(annual_reinsurance) == EXPECTED_CATALOG_YEARS
    and int(np.count_nonzero(annual_reinsurance["catalog_occurrence_count"]))
    == EXPECTED_OCCUPIED_YEARS,
    (
        f"events={len(event_reinsurance)}; annual={len(annual_reinsurance)}; "
        f"occupied={int(np.count_nonzero(annual_reinsurance['catalog_occurrence_count']))}"
    ),
)

maximum_event_error = 0.0
maximum_annual_error = 0.0
maximum_layer_excess = 0.0
for case_name, prefix in CASE_PREFIXES.items():
    event_error = np.abs(
        event_reinsurance[f"{prefix}_gross_insured_loss_2022_usd"]
        - event_reinsurance[
            f"{prefix}_frozen_occurrence_ceded_loss_2022_usd"
        ]
        - event_reinsurance[
            f"{prefix}_frozen_occurrence_retained_loss_2022_usd"
        ]
    )
    maximum_event_error = max(maximum_event_error, float(event_error.max()))
    maximum_layer_excess = max(
        maximum_layer_excess,
        float(
            event_reinsurance[
                f"{prefix}_frozen_occurrence_ceded_loss_2022_usd"
            ].max()
            - FROZEN_LAYER["limit_2022_usd"]
        ),
    )
    for ceded_column, retained_column in [
        (
            f"{prefix}_frozen_occurrence_ceded_aep_2022_usd",
            f"{prefix}_frozen_occurrence_retained_aep_2022_usd",
        ),
        (
            f"{prefix}_standalone_aggregate_ceded_aep_2022_usd",
            f"{prefix}_standalone_aggregate_retained_aep_2022_usd",
        ),
        (
            f"{prefix}_stacked_total_ceded_aep_2022_usd",
            f"{prefix}_stacked_final_retained_aep_2022_usd",
        ),
    ]:
        error = np.abs(
            annual_reinsurance[f"{prefix}_gross_insured_aep_2022_usd"]
            - annual_reinsurance[ceded_column]
            - annual_reinsurance[retained_column]
        )
        maximum_annual_error = max(
            maximum_annual_error, float(error.max())
        )
add_check(
    final_validation_rows,
    "gross_ceded_retained_reconciliation",
    maximum_event_error <= ROW_TOLERANCE_USD
    and maximum_annual_error <= ROW_TOLERANCE_USD,
    (
        f"maximum_event_error={maximum_event_error:.6e}; "
        f"maximum_annual_error={maximum_annual_error:.6e}"
    ),
)
add_check(
    final_validation_rows,
    "frozen_occurrence_layer_limit_obeyed",
    maximum_layer_excess <= ROW_TOLERANCE_USD,
    f"maximum_limit_excess={maximum_layer_excess:.6e}",
)

i0_frozen_row = program_summary.loc[
    program_summary["case_name"].eq("I0_PHASE1_INDEPENDENT")
    & program_summary["program"].eq("FROZEN_OCCURRENCE_XOL")
].iloc[0]
phase1_metric_errors = {
    "ceded_aal_2022_usd": float(i0_frozen_row["ceded_aal_2022_usd"])
    - EXPECTED_PHASE1_REINSURANCE["ceded_aal_2022_usd"],
    "retained_aal_2022_usd": float(i0_frozen_row["retained_aal_2022_usd"])
    - EXPECTED_PHASE1_REINSURANCE["retained_aal_2022_usd"],
    "maximum_ceded_aep_2022_usd": float(
        i0_frozen_row["maximum_ceded_aep_2022_usd"]
    )
    - EXPECTED_PHASE1_REINSURANCE["maximum_ceded_aep_2022_usd"],
}
add_check(
    final_validation_rows,
    "i0_frozen_program_reproduces_phase1",
    max(abs(value) for value in phase1_metric_errors.values())
    <= AGGREGATE_TOLERANCE_USD,
    f"errors_2022_usd={phase1_metric_errors}",
)
add_check(
    final_validation_rows,
    "common_fixed_terms_across_cases",
    program_summary.loc[
        program_summary["program"].eq("FROZEN_OCCURRENCE_XOL"),
        "occurrence_attachment_2022_usd",
    ].nunique()
    == 1
    and program_summary.loc[
        program_summary["program"].eq("FROZEN_OCCURRENCE_XOL"),
        "occurrence_limit_2022_usd",
    ].nunique()
    == 1,
    "The same frozen occurrence attachment and limit are applied to I0, C1, and C2.",
)
add_check(
    final_validation_rows,
    "occurrence_design_grid_complete_and_common",
    len(occurrence_design_grid) == len(occurrence_terms) * len(CASE_PREFIXES)
    and occurrence_design_grid["design_id"].nunique() == len(occurrence_terms),
    (
        f"rows={len(occurrence_design_grid)}; "
        f"designs={occurrence_design_grid['design_id'].nunique()}"
    ),
)
add_check(
    final_validation_rows,
    "aggregate_design_grid_complete_and_ordered",
    len(aggregate_design_grid)
    == len(aggregate_terms) * len(CASE_PREFIXES) * 2
    and set(aggregate_design_grid["structure"])
    == {
        "STANDALONE_ANNUAL_AGGREGATE",
        "STACKED_AFTER_FROZEN_OCCURRENCE",
    },
    (
        f"rows={len(aggregate_design_grid)}; "
        f"structures={sorted(aggregate_design_grid['structure'].unique())}"
    ),
)
add_check(
    final_validation_rows,
    "required_limit_targets_evaluated",
    len(required_limit_summary) == len(CASE_PREFIXES) * 2
    and required_limit_summary["target_kind"].nunique() == 2
    and required_limit_summary.loc[
        normalize_boolean_series(required_limit_summary["feasible"]),
        "achieved_metric_2022_usd",
    ].le(
        required_limit_summary.loc[
            normalize_boolean_series(required_limit_summary["feasible"]),
            "target_metric_2022_usd",
        ]
        + AGGREGATE_TOLERANCE_USD
    ).all(),
    (
        f"rows={len(required_limit_summary)}; "
        f"feasible={int(normalize_boolean_series(required_limit_summary['feasible']).sum())}"
    ),
)
add_check(
    final_validation_rows,
    "pml_tables_nondecreasing",
    all(
        np.all(np.diff(group.sort_values("return_period_years")["pml_2022_usd"]) >= -ROW_TOLERANCE_USD)
        for _, group in pml_table.groupby(
            ["case_name", "program", "loss_basis", "curve_type"], sort=False
        )
    ),
    f"rows={len(pml_table)}",
)
add_check(
    final_validation_rows,
    "var_sparsity_explicitly_flagged",
    program_summary["gross_var_99_2022_usd"].eq(0.0).all()
    and program_summary["gross_var_99_5_2022_usd"].eq(0.0).all()
    and ~normalize_boolean_series(
        program_summary["var_capital_informative_99_5"]
    ).any(),
    "Gross insured AEP VaR is zero at 99.0% and 99.5% because the annual catalog has a dominant zero-loss mass.",
)
add_check(
    final_validation_rows,
    "diversification_metrics_complete",
    len(diversification_summary) == len(CASE_PREFIXES) * 5
    and diversification_summary["units"].eq(EXPECTED_SITES).all()
    and set(diversification_summary["risk_measure"])
    == {"aal", "var", "tvar"},
    f"rows={len(diversification_summary)}; units={sorted(diversification_summary['units'].unique())}",
)
add_check(
    final_validation_rows,
    "paired_uncertainty_complete",
    len(uncertainty_summary)
    == 4 * 2 * len(bootstrap_metrics)
    and uncertainty_summary["bootstrap_replicates"].eq(
        BOOTSTRAP_REPLICATES
    ).all()
    and uncertainty_summary["bootstrap_method"].str.contains("paired").all(),
    (
        f"rows={len(uncertainty_summary)}; "
        f"replicates={BOOTSTRAP_REPLICATES}"
    ),
)
add_check(
    final_validation_rows,
    "raroc_is_assumption_grid_only",
    raroc_assumption_grid["pricing_status"]
    .eq("assumption grid; not a central estimate")
    .all()
    and break_even_premium_grid["pricing_status"]
    .eq("assumption grid; not a quoted price")
    .all(),
    (
        f"raroc_rows={len(raroc_assumption_grid)}; "
        f"break_even_rows={len(break_even_premium_grid)}"
    ),
)
add_check(
    final_validation_rows,
    "public_paths_are_repository_relative",
    all(
        not Path(project_relative_path(path)).is_absolute()
        and "\\" not in project_relative_path(path)
        for path in [
            EVENT_REINSURANCE_PATH,
            ANNUAL_REINSURANCE_PATH,
            INPUT_VALIDATION_PATH,
            CONTROL_VALIDATION_PATH,
            MODEL_SPECIFICATION_PATH,
            PROGRAM_SUMMARY_PATH,
            OCCURRENCE_GRID_PATH,
            AGGREGATE_GRID_PATH,
            PML_TABLE_PATH,
            REQUIRED_LIMIT_PATH,
            DIVERSIFICATION_PATH,
            UNCERTAINTY_PATH,
            BREAK_EVEN_PATH,
            RAROC_PATH,
        ]
    ),
    "All public Notebook 11 paths are repository-relative POSIX paths.",
)
add_check(
    final_validation_rows,
    "extreme_return_period_tail_support_documented",
    False,
    "Return periods with order-statistic rank below 20 are diagnostics, not headline estimates.",
    severity="warning",
)
add_check(
    final_validation_rows,
    "var_based_capital_noninformative_at_selected_confidence",
    False,
    (
        "The specified 99.5% VaR economic-capital measure is non-informative "
        "because gross annual VaR is zero; use the reported TVaR tail capital "
        "and PML metrics for substantive decisions."
    ),
    severity="warning",
)

final_validation = pd.DataFrame(final_validation_rows)
write_csv(final_validation, FINAL_VALIDATION_PATH)
critical_failures = final_validation.loc[
    final_validation["severity"].eq("critical")
    & ~normalize_boolean_series(final_validation["passed"])
]
if not critical_failures.empty:
    display(critical_failures)
    raise RuntimeError("Notebook 11 full validation failed.")

display(
    program_summary[
        [
            "case_name",
            "program",
            "ceded_aal_2022_usd",
            "retained_aal_2022_usd",
            "retained_tvar_99_5_2022_usd",
            "tvar99_5_capital_relief_per_expected_ceded_dollar",
        ]
    ]
)
print("=" * 78)
print("NOTEBOOK 11 CELL 6: REINSURANCE AND CAPITAL VALIDATED")
print("=" * 78)
print(f"Validation checks:            {len(final_validation):,}")
print(f"Critical failures:            {len(critical_failures):,}")
print(f"Documented warnings:          {(final_validation['severity'] == 'warning').sum():,}")
print(f"Event output SHA-256:         {sha256_file(EVENT_REINSURANCE_PATH)}")
print(f"Annual output SHA-256:        {sha256_file(ANNUAL_REINSURANCE_PATH)}")
print("Next: write the portable Notebook 12 parametric-cat-bond handoff.")


In [ ]:
final_validation = pd.read_csv(FINAL_VALIDATION_PATH)
critical_failures = final_validation.loc[
    final_validation["severity"].astype(str).str.lower().eq("critical")
    & ~normalize_boolean_series(final_validation["passed"])
]
if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 11 handoff is blocked by failed critical validation."
    )

artifact_paths = [
    INPUT_VALIDATION_PATH,
    CONTROL_VALIDATION_PATH,
    MODEL_SPECIFICATION_PATH,
    PROGRAM_SUMMARY_PATH,
    OCCURRENCE_GRID_PATH,
    AGGREGATE_GRID_PATH,
    PML_TABLE_PATH,
    REQUIRED_LIMIT_PATH,
    DIVERSIFICATION_PATH,
    UNCERTAINTY_PATH,
    BREAK_EVEN_PATH,
    RAROC_PATH,
    FINAL_VALIDATION_PATH,
    EVENT_REINSURANCE_PATH,
    ANNUAL_REINSURANCE_PATH,
]
artifact_inventory = [
    {
        "path": project_relative_path(path),
        "bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
fixed_results = json.loads(
    program_summary.to_json(orient="records", double_precision=15)
)
required_limit_results = json.loads(
    required_limit_summary.to_json(orient="records", double_precision=15)
)
handoff = {
    "schema_version": SCHEMA_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "notebook11_complete": True,
    "dependence_cases": list(CASE_PREFIXES),
    "frozen_controls": {
        "phase1_release": "v1.0.0",
        "phase1_commit": "be93474ce2ab78d8002d49ae861adb641ae2741d",
        "notebook10_handoff_sha256": sha256_file(NOTEBOOK10_HANDOFF_PATH),
        "phase1_reinsurance_formula_sha256": sha256_lf_normalized_text(
            PHASE1_REINSURANCE_FORMULA_PATH
        ),
        "catalog_years": EXPECTED_CATALOG_YEARS,
        "occurrences": EXPECTED_OCCURRENCES,
        "sites": EXPECTED_SITES,
        "occurrence_layer": FROZEN_LAYER,
    },
    "programs": {
        "fixed": [
            "NO_REINSURANCE",
            "FROZEN_OCCURRENCE_XOL",
            "STANDALONE_AGGREGATE",
            "STACKED_OCCURRENCE_PLUS_AGGREGATE",
        ],
        "stacking_order": (
            "occurrence XoL first; annual aggregate stop-loss second"
        ),
        "occurrence_grid_designs": len(occurrence_terms),
        "aggregate_grid_designs": len(aggregate_terms),
    },
    "tail_metrics": {
        "confidence_levels": list(CONFIDENCE_LEVELS),
        "return_periods_years": list(RETURN_PERIODS),
        "estimator": model_specification["tail_estimator"],
        "var_capital_warning": (
            "99.0% and 99.5% gross annual VaR are zero because zero-loss "
            "years dominate; TVaR tail capital is the substantive supplement."
        ),
    },
    "pricing_boundary": {
        "status": "assumption grid only",
        "premium_multiples_of_gross_aal": PREMIUM_MULTIPLES_OF_GROSS_AAL,
        "expense_ratios": EXPENSE_RATIOS,
        "ceded_price_multipliers": CEDED_PRICE_MULTIPLIERS,
        "no_central_raroc_selected": True,
    },
    "bootstrap": model_specification["bootstrap"],
    "fixed_program_results": fixed_results,
    "required_limit_results": required_limit_results,
    "output_contract": {
        "event_reinsurance": {
            "path": project_relative_path(EVENT_REINSURANCE_PATH),
            "rows": EXPECTED_OCCURRENCES,
            "row_granularity": "one catalog occurrence",
        },
        "annual_reinsurance": {
            "path": project_relative_path(ANNUAL_REINSURANCE_PATH),
            "rows": EXPECTED_CATALOG_YEARS,
            "row_granularity": "one catalog year including zero-event years",
        },
        "program_summary": {
            "path": project_relative_path(PROGRAM_SUMMARY_PATH),
            "rows": len(program_summary),
        },
        "pml_table": {
            "path": project_relative_path(PML_TABLE_PATH),
            "rows": len(pml_table),
        },
        "uncertainty_summary": {
            "path": project_relative_path(UNCERTAINTY_PATH),
            "rows": len(uncertainty_summary),
        },
    },
    "validation": {
        "path": project_relative_path(FINAL_VALIDATION_PATH),
        "sha256": sha256_file(FINAL_VALIDATION_PATH),
        "checks": len(final_validation),
        "critical_failures": 0,
        "warnings": int((final_validation["severity"] == "warning").sum()),
    },
    "artifact_inventory": artifact_inventory,
    "next_notebook": "12_parametric_cat_bond_basis_risk.ipynb",
    "next_task": (
        "Calibrate the frozen magnitude-distance parametric trigger on catalog "
        "years 1-1,000,000, evaluate out-of-sample basis risk on years "
        "1,000,001-2,000,000, and compare identical payouts against I0, C1, "
        "and C2 indemnity targets."
    ),
}
write_json(FINAL_HANDOFF_PATH, handoff)

print("=" * 78)
print("NOTEBOOK 11 COMPLETE: REINSURANCE SENSITIVITY AND CAPITAL VALIDATED")
print("=" * 78)
print(f"Dependence cases:             {len(CASE_PREFIXES):,}")
print(f"Fixed programs:               4")
print(f"Occurrence grid rows:         {len(occurrence_design_grid):,}")
print(f"Aggregate grid rows:          {len(aggregate_design_grid):,}")
print(f"Required-limit rows:          {len(required_limit_summary):,}")
print(f"Validation checks:            {len(final_validation):,}")
print(f"Critical failures:            {len(critical_failures):,}")
print(f"Final handoff:                {project_relative_path(FINAL_HANDOFF_PATH)}")
print("Next: Notebook 12 parametric catastrophe-bond basis risk.")


## Interpretation boundary

Notebook 11 is a technical comparison of identical synthetic
reinsurance programs under different spatial-dependence assumptions.
It is not a market quote, treaty recommendation, or regulatory-capital
calculation.

The 99.0% and 99.5% VaR results must be interpreted with the annual
catalog's dominant zero-loss mass. A zero VaR does not mean that the
portfolio has no tail risk. The corresponding TVaR, PML, and maximum
losses remain material and are reported explicitly.

The RAROC results are scenario surfaces rather than point estimates.
A decision-grade RAROC would require approved earned premium, expense,
brokerage, reinstatement, and reinsurance-pricing inputs.

Notebook 12 will reuse the indemnity targets without changing I0, C1,
or C2 and will evaluate a magnitude-distance parametric payout out of
sample.
